In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt

In [ ]:
import shutil, os

os.makedirs("data", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/data/train.txt", "data/train.txt")

print("파일 로드 완료!")
path = "data/train.txt"
print(f"train.txt 크기: {os.path.getsize(path) / 1024 / 1024:.1f} MB")

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
import sys
sys.path.append('/content/korean-chatbot')
from src.tokenizer import load_tokenizer
from src.model import Transformer

tok = load_tokenizer()
model = Transformer()
model = model.to(device)
print(f"vocab size: {tok.vocab_size}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

class TextDataset(Dataset):
    def __init__(self, path, tokenizer, max_seq_len=512):
        self.samples = []
        
        with open(path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]
        
        print(f"총 {len(lines):,}개 문장 토크나이징 중...")
        
        encoded = tokenizer(
            lines,
            truncation=True,
            max_length=max_seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        for i in tqdm(range(len(lines)), desc="데이터셋 생성 중..."):
            self.samples.append(encoded["input_ids"][i])
        
        print(f"총 샘플 수: {len(self.samples):,}")
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

dataset = TextDataset("data/train.txt", tok)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)  # KoGPT2는 lr 작게



In [ ]:
EPOCHS = 3
for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for step, batch in progress_bar:
        batch = batch.to(device)
        
        loss = model.loss(batch, tok.pad_token_id)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if step % 10 == 0:
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
    
    avg_loss = total_loss / len(dataloader)
    print(f"✨ Epoch {epoch+1} 완료 | 평균 loss: {avg_loss:.4f}\n")

print("🎉 모든 학습 완료!")

In [ ]:
import shutil

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/model.pt")

os.makedirs("/content/drive/MyDrive/korean-chatbot/models", exist_ok=True)
shutil.copy("models/model.pt", "/content/drive/MyDrive/korean-chatbot/models/model.pt")
print("✅ 모델 저장 & Drive 백업 완료!")